# End-to-End Data Engineering Pipeline (Medallion Architecture)
Bronze → Silver → Gold with Data Quality, Feature Engineering, and ML

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


## Load Dataset (Bronze Layer)

In [ ]:
# Replace with your dataset path
df = pd.read_csv('telco_customer_churn.csv')
df.head()

## Bronze Layer: Raw Ingestion

In [ ]:
bronze_df = df.copy()
print("Bronze Shape:", bronze_df.shape)

## Silver Layer: Cleaning & Standardization

In [ ]:
silver_df = bronze_df.copy()

# Example cleaning
silver_df = silver_df.drop_duplicates()

# Handle missing values
silver_df.fillna(method='ffill', inplace=True)

print("Silver Shape:", silver_df.shape)

## Data Quality Validation (Silver → Gold)

In [ ]:
# Schema validation
print(silver_df.dtypes)

# Null check
print("Null Values:\n", silver_df.isnull().sum())

# Simple anomaly detection
numeric_cols = silver_df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    print(col, "Outliers:", ((silver_df[col] > silver_df[col].mean() + 3*silver_df[col].std()) | 
                             (silver_df[col] < silver_df[col].mean() - 3*silver_df[col].std())).sum())

## Feature Engineering (Silver → Gold)

In [ ]:
gold_df = silver_df.copy()

# Example feature: TenureGroup
if 'tenure' in gold_df.columns:
    gold_df['TenureGroup'] = pd.cut(gold_df['tenure'], 
                                   bins=[0,12,24,48,60,100], 
                                   labels=['0-12','12-24','24-48','48-60','60+'])

# Example ratio
if 'MonthlyCharges' in gold_df.columns and 'tenure' in gold_df.columns:
    gold_df['MonthlyCharges_Tenure_Ratio'] = gold_df['MonthlyCharges'] / (gold_df['tenure'] + 1)

gold_df.head()

## Gold Layer: Store in SQLite

In [ ]:
conn = sqlite3.connect('gold_layer.db')
gold_df.to_sql('customer_features', conn, if_exists='replace', index=False)
conn.close()

print("Gold data stored in SQLite!")

## Feature Store Simulation

In [ ]:
feature_store = gold_df[['TenureGroup']].copy()
feature_store['Last_Updated'] = datetime.now().date()
feature_store.to_csv('feature_store.csv', index=False)

feature_store.head()

## ML Model (Optional)

In [ ]:
if 'Churn' in gold_df.columns:
    df_model = gold_df.copy()

    # Encode categorical
    for col in df_model.select_dtypes(include='object').columns:
        df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

    X = df_model.drop('Churn', axis=1)
    y = df_model['Churn']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))